In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
file_path = "../../../data/parquet/games_fully_vectorized.parquet"
df = pd.read_parquet(file_path)

In [3]:
user_input = input("Enter 2 to 5 game names, separated by commas: ")
input_games = [g.strip() for g in user_input.split(',')]

features_df = df.drop(columns=['index', 'name'], errors='ignore')
features_df = features_df.select_dtypes(include=['number', 'bool'])

features_df = features_df.astype(float).fillna(0)

valid_games = []
feature_vectors = []

for game in input_games:
    matched = df[df['name'].str.lower() == game.lower()]
    if not matched.empty:
        valid_games.append(matched.iloc[0]['name'])
        vector = features_df.loc[matched.index[0]].values
        feature_vectors.append(vector)
    else:
        print(f"Warning: Game '{game}' not found in the dataset.")

if not valid_games:
    print("\nError: No valid games found from your input. Please try again.")
else:
    print(f"\nBuilding recommendations based on: {', '.join(valid_games)}...\n")
    
    user_profile = np.mean(feature_vectors, axis=0).reshape(1, -1)
    
    similarities = cosine_similarity(user_profile, features_df.values)[0]
    
    sim_df = pd.DataFrame({
        'name': df['name'],
        'similarity': similarities
    })
    
    sim_df = sim_df[~sim_df['name'].isin(valid_games)]
    
    top_10 = sim_df.sort_values(by='similarity', ascending=False).head(10)
    
    print("--- Top 10 Recommended Games ---")
    for i, (_, row) in enumerate(top_10.iterrows(), 1):
        print(f"{i}. {row['name']} (Similarity: {row['similarity']:.4f})")


Building recommendations based on: Call of Duty: United Offensive, Call of Duty® 2, Grey Zone, Supposedly Wonderful Future...

--- Top 10 Recommended Games ---
1. Dr. Byte (Similarity: 0.8167)
2. Project Robotomania (Similarity: 0.8167)
3. Ode to a Moon (Similarity: 0.8167)
4. Iceburg (Similarity: 0.7976)
5. Convoy Mod Tools (Similarity: 0.7952)
6. Little Witches vs Monsters (Similarity: 0.7952)
7. Better Late Than DEAD (Similarity: 0.7941)
8. Journey to the West(暴躁西游) (Similarity: 0.7926)
9. The Disasters​ (Similarity: 0.7924)
10. Rocket Panda Panic (Similarity: 0.7913)
